# Solutions — JSON, Configuration & Environment Variables

Exercise notebook: `06_files_json_csv/03_json_config_env.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
INCOMING = DATA / "incoming"
print(sorted(p.name for p in INCOMING.iterdir()))

## Exercise 1 — 🟢 Easy · JSON Lines round trip

**Working solution**

In [ ]:
import json
from pathlib import Path

def write_jsonl(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
            count += 1
    return count

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

**Explanation**

JSONL is append-friendly and streamable (Spark, Athena, BigQuery all read it). `default=str` is the escape hatch for non-JSON types.

**Complexity:** O(n).

**Common mistakes**

- `json.dump(list_of_records)` for big data — one giant array can't be streamed or appended.

## Exercise 2 — 🟡 Medium · Config file + environment overrides with type coercion

**Working solution**

In [ ]:
import json
import os
from pathlib import Path

_TRUE, _FALSE = {"true", "1", "yes"}, {"false", "0", "no"}

def _coerce(raw: str, template):
    if isinstance(template, bool):
        low = raw.strip().lower()
        if low in _TRUE:
            return True
        if low in _FALSE:
            return False
        raise ValueError(f"not a boolean: {raw!r}")
    if isinstance(template, int):
        return int(raw)
    if isinstance(template, float):
        return float(raw)
    if isinstance(template, list):
        return [item.strip() for item in raw.split(",") if item.strip()]
    return raw

def load_config(path, env=None, prefix="LAB_"):
    env = os.environ if env is None else env
    config = json.loads(Path(path).read_text(encoding="utf-8"))
    for key, value in config.items():
        var = prefix + key.upper()
        if var in env:
            config[key] = _coerce(env[var], value)
    return config

**Explanation**

Check `bool` **before** `int` — `isinstance(True, int)` is `True` in Python. Passing `env` explicitly makes the function pure and testable;
defaulting to `os.environ` keeps it convenient.

**Complexity:** O(keys).

**Common mistakes**

- `bool('false')` is `True`!
- Reading `os.environ` deep inside business logic, making tests rely on global state.

## Exercise 3 — 🟡 Medium · Customer summary from JSON

**Working solution**

In [ ]:
import json
from collections import Counter
from pathlib import Path

def summarize_customers(path, output_path):
    customers = json.loads(Path(path).read_text(encoding="utf-8"))
    dates = [c["signup_date"] for c in customers if c.get("signup_date")]
    summary = {
        "total": len(customers),
        "active": sum(1 for c in customers if str(c.get("is_active")).lower() == "true"),
        "by_country": dict(sorted(Counter(c.get("country") for c in customers).items())),
        "by_segment": dict(sorted(Counter(c.get("segment") for c in customers).items())),
        "signup_range": [min(dates, default=None), max(dates, default=None)],
    }
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
    return summary

**Explanation**

`sum(1 for ...)` counts without building a list; `Counter` over a generator; `min/max(default=)` for empties. `str(x).lower() == 'true'` normalizes bool-or-string flags.

**Complexity:** O(n).

**Common mistakes**

- `is_active == 'true'` misses real booleans from other producers.

## Exercise 4 — 🟠 Hard · Validate nested JSON against a lightweight schema

**Working solution**

In [ ]:
def _type_name(t):
    return "|".join(x.__name__ for x in (t if isinstance(t, tuple) else (t,)))

def _matches(value, types):
    types = types if isinstance(types, tuple) else (types,)
    if isinstance(value, bool) and bool not in types:
        return False
    return isinstance(value, types)

def validate_json(doc, schema, path="$"):
    if isinstance(schema, dict):
        if not isinstance(doc, dict):
            return [f"{path}: expected object, got {type(doc).__name__}"]
        errors = []
        for raw_key, sub in schema.items():
            optional = raw_key.endswith("?")
            key = raw_key.rstrip("?")
            if key not in doc or (optional and doc[key] is None):
                if not optional:
                    errors.append(f"{path}.{key}: missing")
                continue
            errors += validate_json(doc[key], sub, f"{path}.{key}")
        return errors
    if isinstance(schema, list):
        if not isinstance(doc, list):
            return [f"{path}: expected list, got {type(doc).__name__}"]
        return [e for i, item in enumerate(doc) for e in validate_json(item, schema[0], f"{path}[{i}]")]
    if not _matches(doc, schema):
        return [f"{path}: expected {_type_name(schema)}, got {type(doc).__name__}"]
    return []

**Explanation**

A recursive function that dispatches on the *shape of the schema* mirrors the shape of the data. Carrying a JSONPath-like string gives
actionable errors. (For production use `pydantic` or `jsonschema`; writing a mini one teaches recursion over nested structures.)

**Complexity:** O(size of document).

**Common mistakes**

- Accepting `True` as an `int`.
- Stopping at the first error.

## Challenge — 🔴 Very Hard · Layered settings with provenance

**Working solution**

In [ ]:
import json
import os
from pathlib import Path

_TRUE, _FALSE = {"true", "1", "yes", "on"}, {"false", "0", "no", "off"}

def parse_env_file(text):
    values = {}
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):]
        key, sep, value = line.partition("=")
        if not sep:
            continue
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in "'\"":
            value = value[1:-1]
        elif " #" in value:
            value = value.split(" #", 1)[0].rstrip()
        values[key.strip()] = value
    return values

def _coerce_setting(raw, template):
    if not isinstance(raw, str) or template is None:
        return raw
    if isinstance(template, bool):
        low = raw.lower()
        if low in _TRUE:
            return True
        if low in _FALSE:
            return False
        raise ValueError(f"not a boolean: {raw!r}")
    if isinstance(template, int):
        return int(raw)
    if isinstance(template, float):
        return float(raw)
    if isinstance(template, list):
        return [x.strip() for x in raw.split(",") if x.strip()]
    return raw

class Settings:
    def __init__(self, defaults, json_path=None, env_file=None, environ=None, prefix="LAB_", overrides=None, required=()):
        self._defaults = dict(defaults)
        self._values, self._sources = {}, {}
        self._prefix = prefix
        self._apply(self._defaults, "defaults", coerce=False)
        if json_path:
            self._apply(json.loads(Path(json_path).read_text(encoding="utf-8")), "json", coerce=False)
        if env_file and Path(env_file).exists():
            self._apply(self._normalize(parse_env_file(Path(env_file).read_text(encoding="utf-8"))), "env_file")
        environ = os.environ if environ is None else environ
        self._apply(self._normalize({k: v for k, v in environ.items() if k.startswith(prefix)}), "environ")
        self._apply(overrides or {}, "overrides", coerce=False)
        missing = [k for k in required if k not in self._values]
        if missing:
            raise ValueError(f"missing required settings: {missing}")

    def _normalize(self, mapping):
        return {(k[len(self._prefix):] if k.startswith(self._prefix) else k).lower(): v for k, v in mapping.items()}

    def _apply(self, mapping, source, coerce=True):
        for key, value in mapping.items():
            self._values[key] = _coerce_setting(value, self._defaults.get(key)) if coerce else value
            self._sources[key] = source

    def __getitem__(self, key):
        return self._values[key]

    def get(self, key, default=None):
        return self._values.get(key, default)

    def __contains__(self, key):
        return key in self._values

    def source_of(self, key):
        return self._sources[key]

    def as_dict(self):
        return dict(self._values)

**Explanation**

Each source is normalized to the same shape (lowercase keys, coerced values) and applied in priority order, remembering where each value came from —
invaluable when debugging "why is batch_size 250 in prod?". This is what `pydantic-settings`, Spring's `@ConfigurationProperties`, and Glue's
`getResolvedOptions` + defaults give you, in ~80 lines.

**Complexity:** O(total keys).

**Common mistakes**

- Coercing values from JSON (already typed) — only strings need coercion.
- Leaking unrelated environment variables into settings (use the prefix).